Looking at the celing of N_shots according to Theorem 1 from original IQAE paper

In [11]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.circuit import QuantumCircuit
from qiskit import transpile

qc = QuantumCircuit(1, 1)
qc.x(0)
qc.measure(0, 0)
    
# Build noise model
noise_model = NoiseModel()
error = depolarizing_error(0.75, 1)
noise_model.add_all_qubit_quantum_error(
    error, ['u1', 'u2', 'u3', 'u', 'p', 'sx', 'x', 'rx', 'ry', 'rz']
)

# Noiseless
sim_ideal = AerSimulator()
result_ideal = sim_ideal.run(transpile(qc, sim_ideal), shots=1024).result()
print("Noiseless     :", result_ideal.get_counts())

# Noisy
sim_noisy = AerSimulator(noise_model=noise_model)
result_noisy = sim_noisy.run(transpile(qc, sim_noisy), shots=1024).result()
print("Noisy p=0.75  :", result_noisy.get_counts())

sim_check = AerSimulator()
qc_t = transpile(qc, sim_check)
print("Basis gates used:", qc_t.count_ops())

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json

from qiskit.circuit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import Sampler as AerSampler
from qiskit_algorithms import IterativeAmplitudeEstimation, EstimationProblem

print("All imports OK ✅")

p_true = 0.2

class BernoulliA(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(theta_p, 0)

class BernoulliQ(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(2 * self._theta_p, 0)

    def power(self, k):
        q_k = QuantumCircuit(1)
        q_k.ry(2 * k * self._theta_p, 0)
        return q_k

problem = EstimationProblem(
    state_preparation=BernoulliA(p_true),
    grover_operator=BernoulliQ(p_true),
    objective_qubits=[0],
)
print("Problem defined ✅")

Noiseless     : {'1': 1024}
Noisy p=0.75  : {'0': 375, '1': 649}
Basis gates used: OrderedDict({'x': 1, 'measure': 1})
All imports OK ✅
Problem defined ✅


In [13]:
p_true = 0.2

class BernoulliA(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(theta_p, 0)

class BernoulliQ(QuantumCircuit):
    def __init__(self, probability):
        super().__init__(1)
        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        self.ry(2 * self._theta_p, 0)

    def power(self, k):
        q_k = QuantumCircuit(1)
        q_k.ry(2 * k * self._theta_p, 0)
        return q_k

problem = EstimationProblem(
    state_preparation=BernoulliA(p_true),
    grover_operator=BernoulliQ(p_true),
    objective_qubits=[0],
)
print("Problem defined ✅")

# ══════════════════════════════════════════════════════════════════════════════
# NOISE MODEL BUILDER
# ══════════════════════════════════════════════════════════════════════════════
# All standard single-qubit basis/native gates Qiskit might emit
SINGLE_QUBIT_GATES = [
    'id', 'x', 'y', 'z', 'h', 's', 'sdg', 't', 'tdg',
    'sx', 'sxdg', 'rx', 'ry', 'rz',
    'u', 'u1', 'u2', 'u3', 'p'
]

def build_noisy_sampler(p_depol, n_shots):
    if p_depol > 0.0:
        p_1q = min(p_depol, 0.75)
        error_1q = depolarizing_error(p_1q, 1)

        noise_model = NoiseModel()
        noise_model.add_all_qubit_quantum_error(error_1q, SINGLE_QUBIT_GATES)

        sampler = AerSampler(
            backend_options={"noise_model": noise_model},
            run_options={"shots": n_shots}
        )
    else:
        sampler = AerSampler(run_options={"shots": n_shots})
    return sampler



Problem defined ✅


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Parameters + Helper Functions  (fixed)
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np
from joblib import Parallel, delayed

p_true    = 0.2
R         = 500
ALPHA     = 0.05
N_JOBS    = 18

TARGET         = 0.50
CONV_BAND = 0.02
LOCAL_HALF_WIDTH = 0.02   # accept s ∈ [0.4875, 0.5125]  ← matches agreed pseudocode
CLIFF_THRESH   = 0.05    # deviation > 0.05 from TARGET triggers cliff/drop logic
PRECISION      = 0.0001

epsilon_values = np.linspace(0.005, 0.10, 100).tolist()


# ── Wilson Score ──────────────────────────────────────────────────────────────
Z_95 = 1.96

def wilson_score(k, n, z=Z_95):
    p_hat      = k / n
    z2n        = z**2 / n
    centre     = (p_hat + z2n / 2) / (1 + z2n)
    half_width = (z * np.sqrt(p_hat * (1-p_hat) / n + z2n / 4)) / (1 + z2n)
    lower      = np.clip(centre - half_width, 0.0, 1.0)
    upper      = np.clip(centre + half_width, 0.0, 1.0)
    return float(p_hat), float(lower), float(upper), float(half_width)


# ── Celing N_max ──────────────────────────────────────────────────────────────────
def N_max(epsilon, alpha):
    return int((32 / (1 - 2 * np.sin(np.pi / 14)) ** 2) \
           * np.log((2 / alpha) * np.log2(np.pi / (4 * epsilon))))

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Single run + Probe
# ══════════════════════════════════════════════════════════════════════════════

def single_run(p_noise, eps):
    """One IQAE run. Returns 1 (success) or 0 (failure)."""
    try:
        N_SHOTS = N_max([eps], ALPHA)
        sampler = build_noisy_sampler(p_noise, N_SHOTS)
        iae     = IterativeAmplitudeEstimation(
            epsilon_target = eps,
            alpha          = ALPHA,
            sampler        = sampler,
        )
        result  = iae.estimate(problem)
        # ✅ Use estimation_processed — the final probability in [0,1]
        return 1 if abs(result.estimation_processed - p_true) <= eps else 0
    except Exception as e:
        print(f"[WARN] single_run failed: {type(e).__name__}: {e}")
        return 0


def probe(eps, p):
    """
    Run R independent IQAE runs at (eps, p) in parallel.
    Returns (s_hat, lower_CI, upper_CI, half_width).
    """
    outcomes           = Parallel(n_jobs=N_JOBS)(
        delayed(single_run)(p, eps) for _ in range(R)
    )
    k                  = sum(outcomes)
    s_hat, lo, hi, hw  = wilson_score(k, R)
    return s_hat, lo, hi, hw

In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Bisection
# ══════════════════════════════════════════════════════════════════════════════

def find_boundary(eps, p_low, p_high):
    """
    Bisect over p to find p* where success rate ≈ 50% ± CONV_BAND.
    Matches the pseudocode agreed in transcript exactly.
    """
    history = []

    while (p_high - p_low) > PRECISION:

        p_mid             = (p_low + p_high) / 2.0
        s_hat, lo, hi, hw = probe(eps, p_mid)

        history.append({
            "p_try"       : p_mid,
            "s_hat"       : s_hat,
            "wilson_lower": lo,
            "wilson_upper": hi,
            "wilson_hw"   : hw,
        })

        print(f"    ε={eps:.4f} | p={p_mid:.4f} | "
              f"s={s_hat:.3f} [{lo:.3f}, {hi:.3f}] | "
              f"window=[{p_low:.4f}, {p_high:.4f}]")

        if s_hat > TARGET + CONV_BAND:
            # ✅ Success rate too high → boundary is above p_mid → raise floor
            p_low  = p_mid

        elif s_hat < TARGET - CONV_BAND:
            # ✅ Success rate too low → boundary is below p_mid → lower ceiling
            p_high = p_mid

        else:
            # ✅ In band [0.45, 0.55] → converged
            print(f"    ✅ Converged at p={p_mid:.4f}")
            return {
                "epsilon"     : eps,
                "p_star"      : p_mid,
                "s_hat"       : s_hat,
                "wilson_lower": lo,
                "wilson_upper": hi,
                "wilson_hw"   : hw,
                "converged"   : True,
                "cliff"       : False,
                "history"     : history,
            }

    p_best            = (p_low + p_high) / 2.0
    s_hat, lo, hi, hw = probe(eps, p_best)

    print(f"    ⚠️  Precision limit at p={p_best:.4f} | s={s_hat:.3f}")
    return {
        "epsilon"     : eps,
        "p_star"      : p_best,
        "s_hat"       : s_hat,
        "wilson_lower": lo,
        "wilson_upper": hi,
        "wilson_hw"   : hw,
        "converged"   : False,
        "cliff"       : False,
        "history"     : history,
    }

In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Outer sweep (refined: no auto-skip, local refinement if probe in-band)
# ══════════════════════════════════════════════════════════════════════════════

# Recommended companion parameter settings:
#   CONV_BAND        = 0.0125
#   LOCAL_HALF_WIDTH = 0.02   # or 0.015 for tighter local refinement
#
# Interpretation:
#   - CONV_BAND = 0.0125 means accept only if s_hat is within [0.4875, 0.5125]
#   - LOCAL_HALF_WIDTH defines the half-width of the local p-window searched when
#     the warm-start probe is already close to the boundary. For example,
#     LOCAL_HALF_WIDTH = 0.02 means search in [p_star_prev - 0.02, p_star_prev + 0.02].
#
# This avoids the old behaviour where an in-band warm-start probe was accepted
# immediately and could create sticky plateaus in p*(epsilon).


def sweep_frontier(epsilon_list, local_half_width=LOCAL_HALF_WIDTH):
    """
    Sweep over epsilon values and locate the frontier p*(epsilon).

    Logic:
      - First epsilon: bisect full [0, 1]
      - Subsequent epsilons: probe at previous p*
          * if probe is in-band: do LOCAL refinement around previous p*
          * if probe is too high: boundary moved upward, search [p*_prev, 1]
          * if probe is too low: boundary moved downward, search [0, p*_prev]

    Parameters
    ----------
    epsilon_list : list[float]
        Epsilon grid to sweep.
    local_half_width : float
        Half-width of the local refinement window in p-units. If the probe at
        the previous boundary is already near the 0.5 crossing, search locally
        in [p_star_prev - local_half_width, p_star_prev + local_half_width]
        instead of skipping bisection.
    """
    results = []
    p_star_prev = None

    for eps in epsilon_list:

        print(f"\n{'─'*60}")
        print(f"  ε = {eps:.4f}")
        print(f"{'─'*60}")

        # ── CASE 1: First epsilon — no prior info ────────────────────────────
        if p_star_prev is None:
            p_low = 0.0
            p_high = 1.0
            cliff = False
            print("  First ε — bisecting full range [0.0, 1.0]")

        # ── CASE 2: Subsequent epsilons — probe at previous p* first ────────
        else:
            print(f"  Probing at previous p* = {p_star_prev:.4f} ...")
            s_probe, lo, hi, hw = probe(eps, p_star_prev)
            print(f"  Probe: s={s_probe:.3f} [{lo:.3f}, {hi:.3f}]")

            if abs(s_probe - TARGET) <= CONV_BAND:
                # Near boundary: refine locally instead of auto-accepting.
                p_low = max(0.0, p_star_prev - local_half_width)
                p_high = min(1.0, p_star_prev + local_half_width)
                cliff = False
                print(
                    f"  Near boundary — local refinement in "
                    f"[{p_low:.4f}, {p_high:.4f}] "
                    f"(half-width = {local_half_width:.4f})"
                )

            elif s_probe > TARGET + CONV_BAND:
                # Success too high -> true boundary is above previous p*
                p_low = p_star_prev
                p_high = 1.0
                cliff = True
                print(
                    f"  ⚠️  Cliff detected (s={s_probe:.3f}) — "
                    f"searching [{p_low:.4f}, {p_high:.4f}]"
                )

            else:
                # Success too low -> true boundary is below previous p*
                p_low = 0.0
                p_high = p_star_prev
                cliff = False
                print(
                    f"  ⚠️  Boundary dropped (s={s_probe:.3f}) — "
                    f"searching [{p_low:.4f}, {p_high:.4f}]"
                )

        # ── BISECT within the chosen window ──────────────────────────────────
        result = find_boundary(eps, p_low, p_high)
        result["cliff"] = cliff

        # Optional metadata for debugging / plotting diagnostics
        result["search_window_low"] = p_low
        result["search_window_high"] = p_high
        result["used_local_refinement"] = (
            p_star_prev is not None
            and abs(result["search_window_low"] - max(0.0, p_star_prev - local_half_width)) < 1e-12
            and abs(result["search_window_high"] - min(1.0, p_star_prev + local_half_width)) < 1e-12
            if p_star_prev is not None else False
        )

        results.append(result)
        p_star_prev = result["p_star"]

        status = "✅ converged" if result["converged"] else "⚠️  precision limit"
        print(f"\n  {status} | p* = {result['p_star']:.4f} | s = {result['s_hat']:.3f}")

    return results


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Full run
# ══════════════════════════════════════════════════════════════════════════════

frontier_results = sweep_frontier(epsilon_values)

print(f"\n{'═'*60}")
print(f"  ✅ Frontier sweep complete — {len(frontier_results)} points")
print(f"{'═'*60}")

for r in frontier_results:
    cliff_tag = " ⚠️  CLIFF" if r["cliff"] else ""
    conv_tag  = "✅" if r["converged"] else "⚠️ "
    print(f"  {conv_tag} ε={r['epsilon']:.4f} | "
          f"p*={r['p_star']:.4f} | "
          f"s={r['s_hat']:.3f}{cliff_tag}")


────────────────────────────────────────────────────────────
  ε = 0.0050
────────────────────────────────────────────────────────────
  First ε — bisecting full range [0.0, 1.0]
    ε=0.0050 | p=0.5000 | s=0.000 [0.000, 0.089] | window=[0.0000, 1.0000]
    ε=0.0050 | p=0.2500 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.5000]
    ε=0.0050 | p=0.1250 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.2500]
    ε=0.0050 | p=0.0625 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.1250]
    ε=0.0050 | p=0.0312 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.0625]
    ε=0.0050 | p=0.0156 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.0312]
    ε=0.0050 | p=0.0078 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.0156]
    ε=0.0050 | p=0.0039 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.0078]
    ε=0.0050 | p=0.0020 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.0039]
    ε=0.0050 | p=0.0010 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.0020]
    ε=0.0050 | p=0.0005 | s=0.000 [0.000, 0.089] | window=[0.0000, 0.0

KeyboardInterrupt: 